# Decision Trees & the Gini Impurity
### Business Analytics — Step-by-Step Walkthrough

**Learning Objectives**
1. Understand what a Decision Tree does — and *why* it splits where it does
2. Calculate the **Gini Impurity** by hand (and verify with sklearn)
3. Trace every split in a tree trained on the *California Housing* dataset
4. Visualise the tree, its splits, and the feature importances
5. Evaluate generalisation with a held-out test set

---
> **Dataset note** — We use `sklearn`'s *California Housing* dataset (20,640 districts, 8 features).
> The target is **median house value**, which we bin into three classes:
> `Low`, `Medium`, `High`. This keeps the focus on *classification* trees.


## 0 · Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Nicer plots
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

print("All imports successful ✓")

---
## 1 · Load & Prepare Data

In [ ]:
# --- Load ---
raw = fetch_california_housing(as_frame=True)
df = raw.frame.copy()

# --- Create classification target ---
# Median house value is in $100k units → bin into Low / Medium / High
bins   = [0, 1.5, 3.0, float('inf')]
labels = ['Low', 'Medium', 'High']
df['PriceClass'] = pd.cut(df['MedHouseVal'], bins=bins, labels=labels)

print("Class distribution:")
print(df['PriceClass'].value_counts().sort_index())
print(f"\nShape: {df.shape}")
df.head(3)

In [ ]:
# Quick visual of class balance
fig, ax = plt.subplots(figsize=(5, 3))
df['PriceClass'].value_counts().sort_index().plot(
    kind='bar', ax=ax, color=['#4C72B0','#DD8452','#55A868'], edgecolor='white'
)
ax.set_title('Class Distribution — Median House Value')
ax.set_xlabel('')
ax.set_ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

---
## 2 · The Gini Impurity — Theory & Manual Calculation

The Gini Impurity of a node $t$ with $K$ classes is:

$$
\text{Gini}(t) = 1 - \sum_{k=1}^{K} p_k^2
$$

where $p_k$ is the fraction of samples belonging to class $k$.

**Intuition:** Gini = 0 means *perfectly pure* (all samples belong to one class).  
Gini = 0.667 (for 3 balanced classes) means *maximum impurity* (nothing is certain).

A split is chosen if it **reduces the weighted Gini** of the two child nodes:

$$
\Delta\text{Gini} = \text{Gini}(\text{parent}) - \left[\frac{n_L}{n}\,\text{Gini}(L) + \frac{n_R}{n}\,\text{Gini}(R)\right]
$$

The split with the **largest $\Delta\text{Gini}$** (= information gain) is chosen.

In [ ]:
# ── Helper functions ──────────────────────────────────────────────────────────

def gini(labels):
    """Compute Gini Impurity for a 1-D array of class labels."""
    if len(labels) == 0:
        return 0.0
    classes, counts = np.unique(labels, return_counts=True)
    probs = counts / len(labels)
    return 1.0 - np.sum(probs ** 2)


def weighted_gini(left_labels, right_labels):
    """Weighted Gini of a binary split."""
    n = len(left_labels) + len(right_labels)
    return (len(left_labels) / n) * gini(left_labels) + \
           (len(right_labels) / n) * gini(right_labels)


def best_split(feature_values, labels):
    """
    Brute-force search: find the threshold on `feature_values`
    that maximises Gini gain (= minimises weighted child Gini).
    Returns (best_threshold, best_gain, gain_series).
    """
    thresholds = np.unique(feature_values)
    parent_gini = gini(labels)
    results = []
    for t in thresholds[:-1]:   # skip the last — right child would be empty
        left  = labels[feature_values <= t]
        right = labels[feature_values >  t]
        wg    = weighted_gini(left, right)
        results.append({'threshold': t, 'weighted_gini': wg, 'gain': parent_gini - wg})
    df_res = pd.DataFrame(results)
    best_idx = df_res['gain'].idxmax()
    return (
        df_res.loc[best_idx, 'threshold'],
        df_res.loc[best_idx, 'gain'],
        df_res
    )

print("Helper functions defined ✓")

In [ ]:
# ── Demonstrate Gini on a toy example first ───────────────────────────────────

toy_cases = {
    'Pure node (all Low)'    : np.array(['Low']*10),
    'Two-class 50/50'        : np.array(['Low']*5 + ['High']*5),
    'Three-class balanced'   : np.array(['Low']*4 + ['Medium']*4 + ['High']*4),
    'Almost pure (9/1)'      : np.array(['Low']*9 + ['High']*1),
}

print(f"{'Scenario':<30}  {'Gini':>6}")
print("-" * 40)
for name, arr in toy_cases.items():
    print(f"{name:<30}  {gini(arr):.4f}")

In [ ]:
# ── Visualise: Gini as a function of class proportion (binary case) ───────────

p = np.linspace(0, 1, 300)
gini_binary = 1 - p**2 - (1-p)**2

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(p, gini_binary, lw=2.5, color='#4C72B0', label='Gini (binary)')
ax.axvline(0.5, ls='--', color='#DD8452', label='Max impurity at p=0.5')
ax.set_xlabel('Proportion of class 1  (p)')
ax.set_ylabel('Gini Impurity')
ax.set_title('Gini Impurity — Binary Classification')
ax.legend()
plt.tight_layout()
plt.show()

---
## 3 · Manual Split Search on One Feature

Before we let `sklearn` train the full tree, let's **reproduce the first split by hand**.  
We pick `MedInc` (median income) — which is typically the most important feature.

In [ ]:
# Use a 2,000-row sample for speed (same random state for reproducibility)
sample = df.dropna(subset=['PriceClass']).sample(2000, random_state=SEED)

feature_vals = sample['MedInc'].values
class_labels = sample['PriceClass'].astype(str).values

print(f"Parent node Gini: {gini(class_labels):.4f}")
print(f"Class counts: {pd.Series(class_labels).value_counts().to_dict()}")

In [ ]:
# Downsample thresholds to 200 evenly spaced values (otherwise slow in notebook)
thresholds_full = np.unique(feature_vals)
thresh_idx      = np.linspace(0, len(thresholds_full)-2, 200, dtype=int)
thresholds_sub  = thresholds_full[thresh_idx]

parent_g = gini(class_labels)
rows = []
for t in thresholds_sub:
    left  = class_labels[feature_vals <= t]
    right = class_labels[feature_vals >  t]
    wg    = weighted_gini(left, right)
    rows.append({'threshold': t, 'weighted_gini': wg, 'gain': parent_g - wg})

split_df   = pd.DataFrame(rows)
best_row   = split_df.loc[split_df['gain'].idxmax()]

print(f"Best threshold  : MedInc ≤ {best_row['threshold']:.4f}")
print(f"Weighted Gini   : {best_row['weighted_gini']:.4f}")
print(f"Gini Gain       : {best_row['gain']:.4f}")

In [ ]:
# ── Plot Gini gain across all thresholds ──────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: weighted Gini
axes[0].plot(split_df['threshold'], split_df['weighted_gini'],
             color='#4C72B0', lw=1.8)
axes[0].axvline(best_row['threshold'], color='#DD8452', ls='--',
                label=f"Best split @ {best_row['threshold']:.2f}")
axes[0].axhline(parent_g, color='gray', ls=':', label=f'Parent Gini = {parent_g:.3f}')
axes[0].set_xlabel('MedInc threshold')
axes[0].set_ylabel('Weighted child Gini')
axes[0].set_title('Weighted Gini per Threshold')
axes[0].legend(fontsize=8)

# Right: Gini gain
axes[1].plot(split_df['threshold'], split_df['gain'],
             color='#55A868', lw=1.8)
axes[1].axvline(best_row['threshold'], color='#DD8452', ls='--',
                label=f"Max gain = {best_row['gain']:.4f}")
axes[1].set_xlabel('MedInc threshold')
axes[1].set_ylabel('Gini Gain')
axes[1].set_title('Gini Gain per Threshold  →  pick the peak')
axes[1].legend(fontsize=8)

plt.suptitle('Finding the Best Split on MedInc (manual)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

### Inspect the two child nodes after the split

In [ ]:
t_best  = best_row['threshold']
left_y  = class_labels[feature_vals <= t_best]
right_y = class_labels[feature_vals >  t_best]

for name, subset in [('Left  (MedInc ≤ %.3f)' % t_best, left_y),
                     ('Right (MedInc >  %.3f)' % t_best, right_y)]:
    counts = pd.Series(subset).value_counts().reindex(['Low','Medium','High'], fill_value=0)
    probs  = counts / len(subset)
    g      = gini(subset)
    print(f"{name}")
    print(f"  n={len(subset):5d}  |  ", end='')
    for cls in ['Low','Medium','High']:
        print(f"{cls}: {probs[cls]:.2%}  ", end='')
    print(f"\n  Gini = {g:.4f}\n")

In [ ]:
# ── Bar chart: class distribution in parent vs children ──────────────────────

def class_dist(labels):
    s = pd.Series(labels).value_counts().reindex(['Low','Medium','High'], fill_value=0)
    return s / len(labels)

dist = pd.DataFrame({
    f'Parent\n(n={len(class_labels)})': class_dist(class_labels),
    f'Left child\n(MedInc≤{t_best:.2f}, n={len(left_y)})': class_dist(left_y),
    f'Right child\n(MedInc>{t_best:.2f}, n={len(right_y)})': class_dist(right_y),
})

colors = ['#4C72B0','#DD8452','#55A868']
ax = dist.T.plot(kind='bar', figsize=(8, 4), color=colors, edgecolor='white', width=0.6)
ax.set_title('Class Distribution: Parent → Children after Best Split')
ax.set_ylabel('Proportion')
ax.set_xlabel('')
ax.legend(title='Price Class', bbox_to_anchor=(1, 1))
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

---
## 4 · Compare All Features — Which Gives the Best First Split?

In [ ]:
feature_cols = [c for c in raw.feature_names]  # 8 features

summary = []
for feat in feature_cols:
    vals = sample[feat].values
    thresholds_f = np.unique(vals)
    t_idx = np.linspace(0, len(thresholds_f)-2, min(200, len(thresholds_f)-1), dtype=int)
    best_gain = 0
    best_t    = None
    for t in thresholds_f[t_idx]:
        left  = class_labels[vals <= t]
        right = class_labels[vals >  t]
        gain  = parent_g - weighted_gini(left, right)
        if gain > best_gain:
            best_gain = gain
            best_t    = t
    summary.append({'Feature': feat, 'Best Threshold': best_t, 'Max Gini Gain': best_gain})

summary_df = pd.DataFrame(summary).sort_values('Max Gini Gain', ascending=False)
print(summary_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(
    summary_df['Feature'], summary_df['Max Gini Gain'],
    color='#4C72B0', edgecolor='white'
)
ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=9)
ax.set_xlabel('Max Gini Gain (best threshold per feature)')
ax.set_title('Which Feature Gives the Best Root Split?')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

---
## 5 · Train a Shallow Decision Tree with sklearn

Now we let `sklearn` do the full recursive splitting.  
We start with `max_depth=3` to keep the tree readable.

In [ ]:
# ── Train / Test split ────────────────────────────────────────────────────────

X = df[feature_cols].dropna()
y = df.loc[X.index, 'PriceClass'].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f"Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}")
print("Train class distribution:")
print(y_train.value_counts())

In [ ]:
# ── Fit shallow tree ──────────────────────────────────────────────────────────

clf_shallow = DecisionTreeClassifier(
    criterion   = 'gini',
    max_depth   = 3,
    random_state= SEED
)
clf_shallow.fit(X_train, y_train)

print("Shallow tree (depth=3) trained ✓")
print(f"Train accuracy: {clf_shallow.score(X_train, y_train):.3f}")
print(f"Test  accuracy: {clf_shallow.score(X_test,  y_test):.3f}")

### 5.1 Text representation — every split, sample count, and Gini

In [ ]:
tree_rules = export_text(clf_shallow, feature_names=feature_cols, show_weights=True)
print(tree_rules)

### 5.2 Visual tree — colour-coded by majority class

In [ ]:
class_order = sorted(y_train.unique())   # alphabetical: High, Low, Medium

fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(
    clf_shallow,
    feature_names = feature_cols,
    class_names   = class_order,
    filled        = True,
    rounded       = True,
    impurity      = True,     # <-- show Gini in each node
    proportion    = False,
    fontsize      = 9,
    ax            = ax
)
ax.set_title('Decision Tree (depth=3) — Gini shown in every node', fontsize=14)
plt.tight_layout()
plt.show()

---
## 6 · Verify sklearn's Gini Values by Hand

Let's confirm that the **root node Gini** and the **first split** match what we calculated manually.

In [ ]:
tree_ = clf_shallow.tree_

print("=== Root Node (node 0) ===")
print(f"  sklearn Gini    : {tree_.impurity[0]:.6f}")

# Manually: use full training set
manual_root_gini = gini(y_train.values)
print(f"  Manual Gini     : {manual_root_gini:.6f}")

print()
print("=== First Split ===")
root_feature = feature_cols[tree_.feature[0]]
root_thresh  = tree_.threshold[0]
print(f"  Feature    : {root_feature}")
print(f"  Threshold  : {root_thresh:.6f}")

# Replicate the split
left_mask  = X_train[root_feature] <= root_thresh
right_mask = ~left_mask
g_left  = gini(y_train[left_mask].values)
g_right = gini(y_train[right_mask].values)
n       = len(y_train)
n_l     = left_mask.sum()
n_r     = right_mask.sum()

wg_manual = (n_l/n) * g_left + (n_r/n) * g_right
gain_manual = manual_root_gini - wg_manual

print(f"\n  Left  child — n={n_l:,}  Gini={g_left:.6f}  (sklearn: {tree_.impurity[1]:.6f})")
print(f"  Right child — n={n_r:,}  Gini={g_right:.6f}  (sklearn: {tree_.impurity[2]:.6f})")
print(f"\n  Weighted child Gini : {wg_manual:.6f}")
print(f"  Gini Gain           : {gain_manual:.6f}")

---
## 7 · Overfitting — Depth vs. Accuracy

In [ ]:
depths        = range(1, 20)
train_scores  = []
test_scores   = []

for d in depths:
    clf = DecisionTreeClassifier(criterion='gini', max_depth=d, random_state=SEED)
    clf.fit(X_train, y_train)
    train_scores.append(clf.score(X_train, y_train))
    test_scores.append(clf.score(X_test, y_test))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(depths, train_scores, 'o-', label='Train accuracy', color='#4C72B0')
ax.plot(depths, test_scores,  's-', label='Test accuracy',  color='#DD8452')
ax.axvline(3, ls='--', color='gray', label='Our shallow tree (depth=3)')
ax.set_xlabel('max_depth')
ax.set_ylabel('Accuracy')
ax.set_title('Overfitting Diagnostic — Train vs Test Accuracy')
ax.legend()
plt.tight_layout()
plt.show()

> **Observation:** Train accuracy reaches 100 % for deep trees — the model simply memorises the training data.  
> Test accuracy peaks around depth 5–8 and then declines → classic **overfitting**.

---
## 8 · Feature Importances

In [ ]:
# Use the full-depth best tree (let's pick depth=6 as a sweet spot)
clf_best = DecisionTreeClassifier(criterion='gini', max_depth=6, random_state=SEED)
clf_best.fit(X_train, y_train)

importances = pd.Series(clf_best.feature_importances_, index=feature_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 4))
importances.plot(kind='barh', ax=ax, color='#4C72B0', edgecolor='white')
ax.set_title('Feature Importances (depth=6, Gini-based)')
ax.set_xlabel('Mean decrease in Gini Impurity')
plt.tight_layout()
plt.show()

print(f"\nTest accuracy (depth=6): {clf_best.score(X_test, y_test):.3f}")

---
## 9 · Evaluation — Confusion Matrix & Classification Report

In [ ]:
y_pred = clf_best.predict(X_test)

print(classification_report(y_test, y_pred, target_names=class_order))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels = class_order,
    cmap           = 'Blues',
    ax             = ax
)
ax.set_title('Confusion Matrix — Test Set (depth=6)')
plt.tight_layout()
plt.show()

---
## 10 · Summary & Key Takeaways

| Concept | What we showed |
|---|---|
| **Gini Impurity** | Measures how mixed a node is; 0 = pure, max ≈ 0.67 (3 classes) |
| **Split selection** | Enumerate all thresholds → pick the one with **max Gini Gain** |
| **Feature ranking** | `MedInc` dominates — confirmed manually *and* by sklearn importances |
| **Recursion** | Each child node repeats the search independently |
| **Overfitting** | Deep trees memorise training data; cap depth or use pruning |
| **Verification** | Every sklearn Gini value is reproducible with our `gini()` helper |

### Exercises for Students

1. Change `bins` in Section 1 to create 4 price classes. How does the maximum possible Gini change?
2. Replace `criterion='gini'` with `criterion='entropy'` in Section 5. Compare splits and accuracy.
3. In Section 7, which depth gives the best **test** accuracy? Use `np.argmax` to find it programmatically.
4. Add `min_samples_leaf=50` to the classifier. How does the tree change? Why might this help?
5. Pick a single test observation and trace its path through the tree step-by-step using `tree_.feature` and `tree_.threshold`.
